<a href="https://colab.research.google.com/github/britoadriana/ScreenQA-PT-Eval/blob/main/Notebooks/GIT_EXAMPLE_Qwen_Answer_Classification_ScreenQA_PT_comented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qwen**

Modelo: https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers

In [ ]:
!pip install qwen-vl-utils[decord] transformers

In [ ]:
!pip install datasets

In [ ]:
!pip install -q langchain langchain-core pydantic

**Imports**

In [ ]:
import matplotlib.pyplot as plt
import os
import requests
from PIL import Image
import torch
from transformers import BitsAndBytesConfig
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
from datasets import load_dataset
from typing import Optional
from getpass import getpass
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from google.colab import userdata
from getpass import getpass
import time
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import io
from typing import Literal
from langchain_core.prompts import PromptTemplate

In [ ]:
MODEL_CACHE_DIR = r"/content/drive/MyDrive/qwen"
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
# Criar pasta se não existir
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

# Variaveis que o HF usa para guardar cache, substituindo pelo meu diretório
os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = MODEL_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = MODEL_CACHE_DIR

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
     model_id,
     cache_dir=MODEL_CACHE_DIR,
     dtype="auto",
     device_map="auto",
)

processor = AutoProcessor.from_pretrained(
      model_id,
      cache_dir=MODEL_CACHE_DIR,
)

print("Model loaded!")
print("Model ID:", model_id)

**Model testing**

In [ ]:
conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "http://images.cocodataset.org/val2017/000000039769.jpg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    conversation, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(conversation)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

**Dataset**

In [ ]:
dataset = load_dataset(
    "britoadriana/screen_qa_portuguese",
    split="test",
    cache_dir="/content/drive/MyDrive/screen_qa_portuguese"
)

print(dataset)

**Answers Classification**

In [ ]:
import os
import json
import torch
from tqdm import tqdm
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

# 1. Output Schema Definition
class QuestionClassification(BaseModel):
    category: Literal[
        "OCR",
        "Quantitative Reasoning",
        "Navigation and Interface",
        "Privacy and Access",
        "Location Context",
        "Others"
    ] = Field(description="Functional category of the question.")

    justification: str = Field(
        description="Brief explanation for the category based on the image, question, and ground truth."
    )

# 2. Parser and Prompt Configuration
parser = JsonOutputParser(pydantic_object=QuestionClassification)

SYSTEM_PROMPT = """You are a Ph.D. expert in Natural Language Processing and Computer Vision, with deep knowledge of mobile User Interfaces (UI).
    Your mission is to classify ScreenQA tasks (questions about app screens). You will have access to the question, the correct answer (ground truth), and the screen image.
    The correct answer is provided only as a cognitive clue to identify the type of reasoning required. DO NOT evaluate if the answer is correct or incorrect.
    Classify the task based on one of the 6 functional categories below:

    ### GOLDEN RULES FOR CLASSIFICATION (TAXONOMY):

    1. OCR (Textual Extraction):
       - Criterion: The answer requires ONLY direct text reading. If it involves any logic or searching for icons, use another category.
       - Application: Country names, URLs, phone numbers, version numbers, written languages.

    2. Quantitative Reasoning:
       - Criterion: The answer involves numbers requiring magnitude interpretation, comparison, simple calculations, or COUNTING visual elements.
       - Application: Prices (Currency), percentages, ages, scores, and icon counting (rating stars, "likes", notifications, reviews).
       - Key Difference: If you need to count icons or compare which value is higher/lower, it belongs here, not in OCR.

    3. Navigation and Interface:
       - Criterion: The answer requires understanding "how the screen works," where to click, or identifying element states (colors, buttons, menus).
       - Application: "How-to" instructions, sharing options, Settings, UI colors, and button names.

    4. Privacy and Access (Critical Category):
       - Criterion: Related to user input, credential control, granting system permissions, or sensitive profile data.
       - Application: Login/Signup screens, email/password fields, Permission pop-ups (GPS, Camera), and personally identifiable information (Gender, Personal Data).
       - PRECEDENCE OVER OCR: If the data read (even if text) is used for authentication, user identification, or system permission, you MUST classify it here. An error in this category compromises user integrity.

    5. Location Context:
       - Criterion: Information that situates the device in the real world through the interpretation of geospatial and environmental data.
       - Application: Distances (km/miles), Weather forecasts, maps, and coordinates.

    6. Others:
       - Criterion: Only for "Yes/No" or generic questions, or items that lack clear technical competence in the categories above.

    ### TIE-BREAKING GUIDELINES AND HIERARCHY:
    - PRIORITY 1 (Security): If it involves login, permission, or profile data -> PRIVACY AND ACCESS.
    - PRIORITY 2 (Logic): If it involves comparing values, counting icons, or percentages -> QUANTITATIVE REASONING.
    - PRIORITY 3 (Context): If it involves Weather or Distance -> LOCATION CONTEXT.
    - PRIORITY 4 (Base): If it is just reading a neutral identifier (e.g., version, URL) -> OCR.

{format_instructions}

QUESTION: {question}
GROUND TRUTH: {ground_truth}"""

# 3. Paths and Directories
DRIVE_FOLDER = "/content/drive/MyDrive/results_my_dataset_qwen_screenqa_json"
OUTPUT_FILE = os.path.join(DRIVE_FOLDER, "question_classification_qwen.json")
partial_path = os.path.join(DRIVE_FOLDER, "partial_classification.json")
os.makedirs(DRIVE_FOLDER, exist_ok=True)

# 4. Progress Control
final_results = []
processed_ids = set()

if os.path.exists(partial_path):
    with open(partial_path, "r", encoding="utf-8") as f:
        final_results = json.load(f)
    processed_ids = {item["id"] for item in final_results}
    print(f"Resuming. Items already processed: {len(processed_ids)}")

# 5. Classification Loop
print(f"Starting classification for {len(dataset)} items.")

for i in tqdm(range(len(dataset)), total=len(dataset)):
    if i in processed_ids:
        continue

    item = dataset[i]

    # Adjustment: dataset uses 'answers' (list). We take the first item.
    answers_list = item.get("answers", [""])
    ground_truth_str = answers_list[0] if isinstance(answers_list, list) else str(answers_list)

    full_prompt = SYSTEM_PROMPT.format(
        format_instructions=parser.get_format_instructions(),
        question=item["question"],
        ground_truth=ground_truth_str
    )

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": full_prompt},
                {"type": "image"},
            ],
        },
    ]

    try:
        # Qwen Processing
        text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(
            images=item["image"],
            text=text_prompt,
            return_tensors='pt'
        ).to(model.device, torch.float16)

        output_ids = model.generate(**inputs, max_new_tokens=250, do_sample=False)
        generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
        raw_response = processor.decode(generated_ids, skip_special_tokens=True).strip()

        # Pydantic Parsing
        parsed_output = parser.parse(raw_response)
        category = parsed_output.get("category", "Others")
        justification = parsed_output.get("justification", "")

    except Exception as e:
        category = "PARSING_ERROR"
        justification = f"Raw text: {raw_response} | Error: {str(e)}"

    # Store data (including resource_id for future tracking)
    final_results.append({
        "id": i,
        "resource_id": item.get("resource_id", ""),
        "question": item["question"],
        "ground_truth": ground_truth_str,
        "qwen_question_category": category,
        "qwen_question_justification": justification
    })

    # Partial save every 20 items
    if len(final_results) % 20 == 0:
        with open(partial_path, 'w', encoding='utf-8') as f:
            json.dump(final_results, f, ensure_ascii=False, indent=2)

# 6. Final Save
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print(f"Completed! Final file generated: {OUTPUT_FILE}")